In [1]:
import os, sys, pathlib

# Absolute path to your project root
project_root = pathlib.Path("/home/brian_bosho/FP/FP/federated-gnn")

# Option A: add project root to sys.path (recommended)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Option B (optional): also set the notebook's working directory to project root
# os.chdir(project_root)

# Verify
print("CWD:", os.getcwd())
print("Has project root in sys.path:", str(project_root) in sys.path)

CWD: /home/brian_bosho/FP/FP/federated-gnn/src
Has project root in sys.path: True


In [2]:
import torch


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
# Sanity test with Cora (Planetoid)
import torch
from src.dataprocessing.datasets import GraphDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
gd = GraphDataset(device=DEVICE)

data_cora, ds_cora = gd.load_dataset("Cora", DEVICE)

print("Cora:", data_cora)
print("x:", tuple(data_cora.x.shape), 
      "y:", tuple(data_cora.y.shape), 
      "edges:", tuple(data_cora.edge_index.shape))

# Basic mask checks
print("masks (train/val/test):", 
      int(data_cora.train_mask.sum()),
      int(data_cora.val_mask.sum()),
      int(data_cora.test_mask.sum()))

# Assertions
assert data_cora.edge_index.shape[0] == 2, "edge_index must have 2 rows"
assert data_cora.train_mask.dtype == torch.bool, "train_mask must be boolean"
assert data_cora.val_mask.dtype == torch.bool, "val_mask must be boolean"
assert data_cora.test_mask.dtype == torch.bool, "test_mask must be boolean"

# No overlaps between masks
overlap_tv = (data_cora.train_mask & data_cora.val_mask).sum().item()
overlap_tt = (data_cora.train_mask & data_cora.test_mask).sum().item()
overlap_vt = (data_cora.val_mask & data_cora.test_mask).sum().item()
assert overlap_tv == 0 and overlap_tt == 0 and overlap_vt == 0, "Masks must be disjoint"

# Optional: ensure labels shape is 1-D
if hasattr(data_cora, "y") and len(data_cora.y.shape) > 1 and data_cora.y.shape[1] == 1:
    raise AssertionError("Expected y to be 1-D after loading")

Cora: Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
x: (2708, 1433) y: (2708,) edges: (2, 10556)
masks (train/val/test): 140 500 1000


In [5]:
# Test Amazon Computers and Photo
import torch
from src.dataprocessing.datasets import GraphDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
gd = GraphDataset(device=DEVICE)

# Computers
data_computers, ds_computers = gd.load_dataset("Computers", DEVICE)
print("Computers:", data_computers)
print("x:", tuple(data_computers.x.shape), "y:", tuple(data_computers.y.shape), "edges:", tuple(data_computers.edge_index.shape))
print("masks (train/val/test):",
      int(data_computers.train_mask.sum()),
      int(data_computers.val_mask.sum()),
      int(data_computers.test_mask.sum()))
assert data_computers.edge_index.shape[0] == 2
assert data_computers.train_mask.dtype == torch.bool

# Photo
data_photo, ds_photo = gd.load_dataset("Photo", DEVICE)
print("Photo:", data_photo)
print("x:", tuple(data_photo.x.shape), "y:", tuple(data_photo.y.shape), "edges:", tuple(data_photo.edge_index.shape))
print("masks (train/val/test):",
      int(data_photo.train_mask.sum()),
      int(data_photo.val_mask.sum()),
      int(data_photo.test_mask.sum()))
assert data_photo.edge_index.shape[0] == 2
assert data_photo.train_mask.dtype == torch.bool


Computers: Data(x=[13752, 767], edge_index=[2, 505474], y=[13752], train_mask=[13752], val_mask=[13752], test_mask=[13752])
x: (13752, 767) y: (13752,) edges: (2, 505474)
masks (train/val/test): 12252 500 1000
Photo: Data(x=[7650, 745], edge_index=[2, 245812], y=[7650], train_mask=[7650], val_mask=[7650], test_mask=[7650])
x: (7650, 745) y: (7650,) edges: (2, 245812)
masks (train/val/test): 6150 500 1000


In [6]:
# Setup: ensure project root on sys.path (if not already done)
import os, sys, pathlib
project_root = pathlib.Path("/home/brian_bosho/FP/FP/federated-gnn")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("CWD:", os.getcwd())
print("Has project root in sys.path:", str(project_root) in sys.path)

CWD: /home/brian_bosho/FP/FP/federated-gnn/src
Has project root in sys.path: True


In [7]:
# Fix for full_dataset in notebook: call low-level loader directly with device
import torch
from src.run import load_data, load_configuration
from src.dataprocessing.loaders import load_dataset as low_level_load_dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_clients, beta, cfg = load_configuration("/home/brian_bosho/FP/FP/federated-gnn/conf/base.yaml")

def summarize_full(data, dataset):
    print(f"Full graph: nodes={data.num_nodes}, edges={data.edge_index.shape[1]}")
    print(f"x={tuple(data.x.shape)} y={tuple(data.y.shape)}")
    if hasattr(data, "train_mask"):
        print("masks (train/val/test):",
              int(data.train_mask.sum()),
              int(data.val_mask.sum()),
              int(data.test_mask.sum()))

def summarize_split(data, dataset, clients_data, test_data):
    print(f"Full graph: nodes={data.num_nodes}, edges={data.edge_index.shape[1]}")
    print(f"x={tuple(data.x.shape)} y={tuple(data.y.shape)}")
    if hasattr(data, "train_mask"):
        print("masks (train/val/test):",
              int(data.train_mask.sum()),
              int(data.val_mask.sum()),
              int(data.test_mask.sum()))
    print(f"Num clients: {len(clients_data)}")
    print("Client[0]:",
          f"nodes={clients_data[0].num_nodes}, edges={clients_data[0].edge_index.shape[1]}",
          f"x={tuple(clients_data[0].x.shape)}")

def run_option(dataset_name, option):
    print(f"\n=== {dataset_name} | option={option} ===")
    if option == "full_dataset":
        # Call the underlying loader directly with device
        data, dataset = low_level_load_dataset(dataset_name, DEVICE)
        summarize_full(data, dataset)
        return data, dataset, None, None
    else:
        data, dataset, clients_data, test_data = load_data(
            option, num_clients, beta, dataset_name, DEVICE, config=cfg
        )
        summarize_split(data, dataset, clients_data, test_data)
        return data, dataset, clients_data, test_data

In [8]:
run_option("Photo", "diffusion")



=== Photo | option=diffusion ===
Feature propagation logs saved to: /home/brian_bosho/FP/FP/federated-gnn/logs/propagation_stats/prop_exp_20251006-095712_diffusion_beta_1_hop_1.json
Full graph: nodes=7650, edges=245812
x=(7650, 1833) y=(7650,)
masks (train/val/test): 6150 500 1000
Num clients: 10
Client[0]: nodes=3305, edges=98511 x=(3305, 1833)


(Data(x=[7650, 1833], edge_index=[2, 245812], y=[7650], train_mask=[7650], val_mask=[7650], test_mask=[7650]),
 AmazonPhoto(),
 [Data(x=[3305, 1833], edge_index=[2, 98511], y=[3305], train_mask=[3305], val_mask=[3305], test_mask=[3305], mapping=[300], original_feature_dim=745, num_features=1833),
  Data(x=[4042, 1833], edge_index=[2, 120124], y=[4042], train_mask=[4042], val_mask=[4042], test_mask=[4042], mapping=[810], original_feature_dim=745, num_features=1833),
  Data(x=[5002, 1833], edge_index=[2, 190458], y=[5002], train_mask=[5002], val_mask=[5002], test_mask=[5002], mapping=[565], original_feature_dim=745, num_features=1833),
  Data(x=[4723, 1833], edge_index=[2, 173531], y=[4723], train_mask=[4723], val_mask=[4723], test_mask=[4723], mapping=[552], original_feature_dim=745, num_features=1833),
  Data(x=[5732, 1833], edge_index=[2, 208072], y=[5732], train_mask=[5732], val_mask=[5732], test_mask=[5732], mapping=[808], original_feature_dim=745, num_features=1833),
  Data(x=[5555

In [9]:
# clear cuda
torch.cuda.empty_cache()

In [ ]:
run_option("Computers", "diffusion")

In [ ]:
run_option("Photo", "diffusion")
